In [1]:

# =============================================================================
# CELL 1 — setup, data, deduplication
# =============================================================================
 
import os, re, glob, json, time, zipfile, hashlib, urllib.request
import numpy as np, pandas as pd, librosa, torch
from scipy.signal import butter, lfilter
from transformers import AutoFeatureExtractor, WavLMModel
 
SR, WIN_S = 16000, 4.0
WIN = int(SR * WIN_S)
TARGET_RMS = 0.05
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
 
fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").to(dev).eval()
for p in wavlm.parameters():
    p.requires_grad = False
if dev == "cuda":
    wavlm = wavlm.half()
 
 
def load_norm(path):
    """One loader. Crop to 4s, never pad."""
    wav, _ = librosa.load(path, sr=SR, mono=True)
    rms = np.sqrt(np.mean(wav ** 2)) if len(wav) else 0.0
    if rms > 1e-8:
        wav = wav * min(TARGET_RMS / rms, 10.0)
        peak = np.abs(wav).max()
        if peak > 0.99:
            wav = wav * (0.99 / peak)
    return wav[:WIN].astype(np.float32)
 
 
def degrade(wav, seed=0, sr=SR):
    """Channel augmentation only. The corpus gap is no longer its job."""
    r = np.random.default_rng(seed)
    b, a = butter(6, r.uniform(3000, 7000) / (sr / 2), btype="low")
    w = lfilter(b, a, wav)
    sp = np.mean(w ** 2)
    if sp > 1e-10:
        w = w + r.normal(0, np.sqrt(sp / 10 ** (r.uniform(5, 25) / 10)), len(w))
    w = w * r.uniform(0.5, 1.5)
    peak = np.abs(w).max()
    return (w * (0.99 / peak) if peak > 0.99 else w).astype(np.float32)
 
 
@torch.no_grad()
def embed(paths, augment=False, tag=""):
    """No padding. One clip at a time, mean-pooled over its own frames."""
    out, t0 = [], time.time()
    for k, p in enumerate(paths):
        w = load_norm(p) if isinstance(p, str) else p.astype(np.float32)
        if augment:
            w = degrade(w, seed=k)
        if len(w) < int(0.2 * SR):
            w = np.pad(w, (0, int(0.2 * SR) - len(w)))
        x = fe([w], sampling_rate=SR, return_tensors="pt").input_values.to(dev)
        if dev == "cuda":
            x = x.half()
        out.append(wavlm(x).last_hidden_state.mean(1).float().cpu().numpy())
        if (k + 1) % 400 == 0 or k + 1 == len(paths):
            print(f"  {tag} {k+1}/{len(paths)} {time.time()-t0:.0f}s")
    return np.vstack(out)
 
 
# ---- VIVAE, deduplicated ----
DST = "/kaggle/working/vivae"
os.makedirs(DST, exist_ok=True)
if not glob.glob(f"{DST}/**/*.wav", recursive=True):
    urllib.request.urlretrieve(
        "https://zenodo.org/records/4066235/files/VIVAE.zip?download=1",
        "/kaggle/working/VIVAE.zip")
    with zipfile.ZipFile("/kaggle/working/VIVAE.zip") as z:
        z.extractall(DST)
 
rows = []
for p in sorted(glob.glob(f"{DST}/**/*.wav", recursive=True)):
    spk, emo, inten, item = os.path.splitext(os.path.basename(p))[0].split("_")
    rows.append({"path": p, "speaker": spk, "emotion": emo, "intensity": inten,
                 "sha": hashlib.sha1(open(p, "rb").read()).hexdigest()})
vd = pd.DataFrame(rows)
n_before = len(vd)
vd = vd.drop_duplicates("sha").reset_index(drop=True)
print(f"VIVAE {n_before} -> {len(vd)} after dedup by content hash")
print(vd.emotion.value_counts().to_string())
 
# ---- CREMA-D ----
EMO = {"ANG": "angry", "DIS": "disgust", "FEA": "fear",
       "HAP": "happy", "NEU": "neutral", "SAD": "sad"}
rows = []
for p in glob.glob("/kaggle/input/**/*.wav", recursive=True):
    b = os.path.basename(p)
    if re.match(r"^\d{4}_[A-Z]{3}_[A-Z]{3}_[A-Z]{2}\.wav$", b):
        a, s, e, i = os.path.splitext(b)[0].split("_")
        rows.append({"path": p, "actor_id": a, "emotion": EMO.get(e, e.lower()),
                     "intensity": i})
cd = pd.DataFrame(rows).drop_duplicates("path").reset_index(drop=True)
print(f"\nCREMA-D {len(cd)} clips, {cd.actor_id.nunique()} actors")
 
# ---- medical ----
seen, med = set(), []
for p in glob.glob("/kaggle/input/**/*.wav", recursive=True):
    if "medical" in p.lower() and os.path.basename(p) not in seen:
        seen.add(os.path.basename(p)); med.append(p)
print(f"medical {len(med)} unique")
 
# ---- ESC-50 ward noise ----
ESC = "/kaggle/working/esc50"
if not glob.glob(f"{ESC}/**/*.wav", recursive=True):
    urllib.request.urlretrieve(
        "https://github.com/karoldvl/ESC-50/archive/master.zip",
        "/kaggle/working/esc50.zip")
    with zipfile.ZipFile("/kaggle/working/esc50.zip") as z:
        z.extractall(ESC)
esc_meta = pd.read_csv(glob.glob(f"{ESC}/**/esc50.csv", recursive=True)[0])
esc_dir = os.path.dirname(glob.glob(f"{ESC}/**/*.wav", recursive=True)[0])
WARD = ["clock_alarm", "door_wood_knock", "footsteps", "clock_tick",
        "washing_machine", "vacuum_cleaner", "mouse_click", "keyboard_typing",
        "can_opening", "door_wood_creaks", "water_drops", "toilet_flush"]
esc_paths = [os.path.join(esc_dir, f)
             for f in esc_meta[esc_meta.category.isin(WARD)].filename]
print(f"ESC-50 ward noise {len(esc_paths)} clips")
 
# ---- synthetic non-speech: the inputs that scored 0.9996 ----
rng = np.random.default_rng(0)
SYNTH = []
for dur in [0.5, 1.0, 2.0, 4.0]:
    SYNTH.append(np.zeros(int(dur * SR), dtype=np.float32))
    for dbfs in [-60, -50, -40]:
        SYNTH.append(rng.normal(0, 10 ** (dbfs / 20), int(dur * SR)).astype(np.float32))
print(f"synthetic non-speech {len(SYNTH)} clips")
 
 

device: cuda


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

VIVAE 1565 -> 1085 after dedup by content hash
emotion
pleasure       202
surprise       187
pain           185
fear           176
anger          174
achievement    161

CREMA-D 7441 clips, 91 actors
medical 6661 unique
ESC-50 ward noise 480 clips
synthetic non-speech 16 clips


In [2]:
# ===== CELL 2 — features (CREMA extracted once, neutral sliced out) =====

DISTRESS = ["pain", "fear"]
OTHER    = ["pleasure", "surprise", "achievement", "anger"]

vpos = vd[vd.emotion.isin(DISTRESS)].reset_index(drop=True)
vneg = vd[vd.emotion.isin(OTHER)].reset_index(drop=True)
mds  = list(np.random.default_rng(0).choice(med, size=min(800, len(med)), replace=False))

F = {}
for name, paths in [("vpos", vpos.path.tolist()), ("vneg", vneg.path.tolist()),
                    ("esc", esc_paths), ("med", mds), ("crema", cd.path.tolist())]:
    F[(name, "clean")] = embed(paths, augment=False, tag=f"{name}/clean")
    F[(name, "aug")]   = embed(paths, augment=True,  tag=f"{name}/aug")

F[("synth", "clean")] = embed(SYNTH, augment=False, tag="synth")
F[("synth", "aug")]   = embed(SYNTH, augment=True,  tag="synth/aug")

# neutral is a slice of crema, not a second extraction
neu_mask = (cd.emotion == "neutral").values
cneu = cd[neu_mask].reset_index(drop=True)
F[("cneu", "clean")] = F[("crema", "clean")][neu_mask]
F[("cneu", "aug")]   = F[("crema", "aug")][neu_mask]

for k, v in F.items():
    print(k, v.shape)

  vpos/clean 361/361 28s
  vpos/aug 361/361 7s
  vneg/clean 400/724 10s
  vneg/clean 724/724 18s
  vneg/aug 400/724 8s
  vneg/aug 724/724 14s
  esc/clean 400/480 14s
  esc/clean 480/480 17s
  esc/aug 400/480 15s
  esc/aug 480/480 18s
  med/clean 400/800 59s
  med/clean 800/800 106s
  med/aug 400/800 17s
  med/aug 800/800 33s
  crema/clean 400/7441 17s
  crema/clean 800/7441 34s
  crema/clean 1200/7441 50s
  crema/clean 1600/7441 67s
  crema/clean 2000/7441 82s
  crema/clean 2400/7441 98s
  crema/clean 2800/7441 114s
  crema/clean 3200/7441 129s
  crema/clean 3600/7441 149s
  crema/clean 4000/7441 163s
  crema/clean 4400/7441 179s
  crema/clean 4800/7441 195s
  crema/clean 5200/7441 212s
  crema/clean 5600/7441 228s
  crema/clean 6000/7441 244s
  crema/clean 6400/7441 259s
  crema/clean 6800/7441 275s
  crema/clean 7200/7441 291s
  crema/clean 7441/7441 302s
  crema/aug 400/7441 11s
  crema/aug 800/7441 21s
  crema/aug 1200/7441 32s
  crema/aug 1600/7441 43s
  crema/aug 2000/7441 54s
  

In [3]:
 
# =============================================================================
# CELL 3 — distress gate, refit within-corpus
# =============================================================================
 
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve
 
def lr():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=3000, class_weight="balanced"))
 
def stack(name):
    return np.vstack([F[(name, "clean")], F[(name, "aug")]])
 
def grp(values):
    return np.r_[values, values]
 
# Positives: VIVAE pain + fear.
# Negatives, primary: VIVAE's own other vocalisations. Same mics, same room,
# same speakers, so the only difference left is what was vocalised.
# Negatives, robustness: CREMA-D neutral, ward noise, silence. These make the
# gate safe outside its training distribution but they do NOT set the
# threshold, because they come from different recording conditions.
 
Xp, Xn = stack("vpos"), stack("vneg")
Xrobust = np.vstack([stack("cneu"), stack("esc"), stack("synth")])
 
X_in = np.vstack([Xp, Xn])
y_in = np.r_[np.ones(len(Xp)), np.zeros(len(Xn))]
g_in = np.r_[grp(vpos.speaker.values), grp(vneg.speaker.values)]
 
print("=== within-corpus, speaker-held-out ===")
oof = cross_val_predict(lr(), X_in, y_in, cv=GroupKFold(5), groups=g_in,
                        method="predict_proba")[:, 1]
auc_in = roc_auc_score(y_in, oof)
print(f"  AUC = {auc_in:.4f}")
 
# Honest per-emotion means: out-of-fold, so no clip scores itself.
emo = np.r_[grp(vpos.emotion.values), grp(vneg.emotion.values)]
print("\nmean out-of-fold score per emotion:")
print(pd.Series(oof, index=emo).groupby(level=0).mean().sort_values().to_string())
 
# ---- threshold, chosen from the within-corpus curve -------------------------
fpr, tpr, thr = roc_curve(y_in, oof)
rows = []
for t in np.quantile(oof, np.linspace(0.5, 0.98, 25)):
    rec = float((oof[y_in == 1] >= t).mean())
    fa_in = float((oof[y_in == 0] >= t).mean())
    fa_ang = float((oof[(y_in == 0) & (emo == "anger")] >= t).mean())
    rows.append({"threshold": round(float(t), 4), "recall": round(rec, 3),
                 "fa_within_corpus": round(fa_in, 3), "fa_anger": round(fa_ang, 3)})
curve = pd.DataFrame(rows)
print("\n=== operating curve, within-corpus negatives ===")
print(curve.to_string(index=False))
 
# Pick the point. Edit this after reading the table above — do not accept the
# default without looking. The old threshold was never chosen, it was inherited
# from a curve built on the wrong negatives.
TARGET_FA = 0.20
pick = curve[curve.fa_within_corpus <= TARGET_FA].iloc[0]
DISTRESS_HIGH_NEW = float(pick.threshold)
print(f"\nchosen: thr={DISTRESS_HIGH_NEW:.4f} recall={pick.recall} "
      f"fa={pick.fa_within_corpus} anger_fa={pick.fa_anger}")
 
# ---- final model: within-corpus negatives PLUS the robustness set -----------
Xf = np.vstack([Xp, Xn, Xrobust])
yf = np.r_[np.ones(len(Xp)), np.zeros(len(Xn) + len(Xrobust))]
distress_final = lr().fit(Xf, yf)
 
print("\n=== false alarms at the new threshold ===")
for name, label in [("med", "medical speech"), ("esc", "ward noise"),
                    ("synth", "silence / noise floor"), ("cneu", "CREMA-D neutral")]:
    for cond in ["clean", "aug"]:
        s = distress_final.predict_proba(F[(name, cond)])[:, 1]
        print(f"  {label:22s} {cond:5s} FA = {(s >= DISTRESS_HIGH_NEW).mean():.3f}")
 
print("\n=== sanity: does it still fire on real distress? ===")
for cond in ["clean", "aug"]:
    s = distress_final.predict_proba(F[("vpos", cond)])[:, 1]
    print(f"  VIVAE pain/fear {cond:5s} recall = {(s >= DISTRESS_HIGH_NEW).mean():.3f}")

=== within-corpus, speaker-held-out ===
  AUC = 0.6293

mean out-of-fold score per emotion:
pleasure       0.244825
achievement    0.316174
surprise       0.331464
anger          0.416613
pain           0.465375
fear           0.479507

=== operating curve, within-corpus negatives ===
 threshold  recall  fa_within_corpus  fa_anger
    0.2295   0.625             0.438     0.546
    0.2574   0.614             0.414     0.526
    0.2917   0.601             0.390     0.497
    0.3256   0.575             0.373     0.474
    0.3637   0.548             0.356     0.454
    0.4038   0.525             0.338     0.448
    0.4519   0.499             0.321     0.431
    0.4981   0.474             0.303     0.405
    0.5366   0.446             0.287     0.391
    0.5910   0.413             0.274     0.382
    0.6417   0.382             0.259     0.356
    0.6786   0.357             0.242     0.330
    0.7160   0.342             0.219     0.313
    0.7609   0.313             0.204     0.302
    0.795

In [5]:
# ===== CELL 3b — held-out evaluation of the final configuration =====
# One model, one CV, so recall and every false-alarm rate come from the same
# held-out evaluation. Noise and synthetic clips get their own groups.

Xall = np.vstack([Xp, Xn, stack("cneu"), stack("esc"), stack("synth")])
yall = np.r_[np.ones(len(Xp)), np.zeros(len(Xn) + len(stack("cneu"))
                                        + len(stack("esc")) + len(stack("synth")))]
gall = np.r_[grp(vpos.speaker.values), grp(vneg.speaker.values),
             grp(cneu.actor_id.values),
             np.array([f"esc_{i%5}"   for i in range(len(stack("esc")))]),
             np.array([f"synth_{i%5}" for i in range(len(stack("synth")))])]
src = np.r_[np.array(["distress"] * len(Xp)), np.array(["vivae_other"] * len(Xn)),
            np.array(["crema_neutral"] * len(stack("cneu"))),
            np.array(["ward_noise"] * len(stack("esc"))),
            np.array(["silence"] * len(stack("synth")))]

oof2 = cross_val_predict(lr(), Xall, yall, cv=GroupKFold(5), groups=gall,
                         method="predict_proba")[:, 1]
print(f"held-out AUC, all negatives: "
      f"{roc_auc_score(yall, oof2):.4f}")

# medical was never in training at all, so score it from a model fit on the rest
med_scores = lr().fit(Xall, yall).predict_proba(stack("med"))[:, 1]

rows = []
for t in np.quantile(oof2[yall == 1], np.linspace(0.05, 0.9, 20)):
    r = {"threshold": round(float(t), 4),
         "recall": round(float((oof2[yall == 1] >= t).mean()), 3),
         "fa_medical": round(float((med_scores >= t).mean()), 4)}
    for s in ["vivae_other", "crema_neutral", "ward_noise", "silence"]:
        m = (src == s)
        r[f"fa_{s}"] = round(float((oof2[m] >= t).mean()), 3)
    rows.append(r)
dep = pd.DataFrame(rows)
print("\n=== deployment curve, all held out ===")
print(dep.to_string(index=False))

held-out AUC, all negatives: 0.8632

=== deployment curve, all held out ===
 threshold  recall  fa_medical  fa_vivae_other  fa_crema_neutral  fa_ward_noise  fa_silence
    0.0024   0.949      0.1044           0.885             0.086          0.190       0.500
    0.0104   0.904      0.0537           0.807             0.036          0.110       0.406
    0.0295   0.860      0.0288           0.734             0.015          0.069       0.281
    0.0696   0.816      0.0163           0.673             0.008          0.042       0.250
    0.1171   0.770      0.0144           0.615             0.006          0.034       0.250
    0.1836   0.726      0.0119           0.562             0.001          0.029       0.156
    0.2679   0.681      0.0088           0.504             0.001          0.024       0.094
    0.4028   0.637      0.0075           0.443             0.000          0.017       0.000
    0.4899   0.591      0.0050           0.408             0.000          0.014       0.000
    

In [4]:
# =============================================================================
# CELL 4 — arousal head, refit with a non-speech class
# =============================================================================
 
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
 
def mlp():
    return make_pipeline(StandardScaler(),
                         MLPClassifier(hidden_layer_sizes=(256,), max_iter=400,
                                       early_stopping=False, alpha=1e-3,
                                       random_state=42))
 
# Seventh class. A fan is not an emotion, and the head needs somewhere to put it.
Xa = np.vstack([stack("crema"), stack("esc"), stack("synth")])
ya = np.r_[grp(cd.emotion.values),
           np.array(["non_speech"] * (len(F[("esc", "clean")]) * 2
                                      + len(F[("synth", "clean")]) * 2))]
 
# Noise clips get unique synthetic groups so no fold loses a class.
ga = np.r_[grp(cd.actor_id.values),
           np.array([f"noise_{i%5}" for i in range(len(ya) - 2 * len(cd))])]
 
le = LabelEncoder()
yai = le.fit_transform(ya)
names = list(le.classes_)
fear_index_new = names.index("fear")
print("classes:", names, " fear_index =", fear_index_new)
 
pred = cross_val_predict(mlp(), Xa, yai, cv=GroupKFold(5), groups=ga)
print(classification_report(yai, pred, target_names=names, digits=3))
 
arousal_final = mlp().fit(Xa, yai)
 
# Threshold from the within-CREMA-D curve, not from medical clips.
oof_proba = cross_val_predict(mlp(), Xa, yai, cv=GroupKFold(5), groups=ga,
                              method="predict_proba")[:, fear_index_new]
is_fear = (ya == "fear")
rows = []
for t in np.quantile(oof_proba, np.linspace(0.5, 0.99, 25)):
    rows.append({"threshold": round(float(t), 4),
                 "fear_recall": round(float((oof_proba[is_fear] >= t).mean()), 3),
                 "fa_within": round(float((oof_proba[~is_fear] >= t).mean()), 3)})
acurve = pd.DataFrame(rows)
print("\n=== arousal operating curve, within CREMA-D ===")
print(acurve.to_string(index=False))
 
AROUSAL_TARGET_FA = 0.10
apick = acurve[acurve.fa_within <= AROUSAL_TARGET_FA].iloc[0]
AROUSAL_HIGH_NEW = float(apick.threshold)
print(f"\nchosen: thr={AROUSAL_HIGH_NEW:.4f} recall={apick.fear_recall} "
      f"fa={apick.fa_within}")
 
print("\n=== arousal false alarms at the new threshold ===")
for name, label in [("med", "medical speech"), ("esc", "ward noise"),
                    ("synth", "silence / noise floor")]:
    for cond in ["clean", "aug"]:
        s = arousal_final.predict_proba(F[(name, cond)])[:, fear_index_new]
        print(f"  {label:22s} {cond:5s} FA = {(s >= AROUSAL_HIGH_NEW).mean():.3f}")

classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'non_speech', 'sad']  fear_index = 2


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


              precision    recall  f1-score   support

       angry      0.784     0.738     0.760      2542
     disgust      0.612     0.603     0.608      2542
        fear      0.555     0.550     0.552      2542
       happy      0.630     0.630     0.630      2542
     neutral      0.696     0.722     0.709      2174
  non_speech      0.988     0.989     0.988       992
         sad      0.547     0.575     0.561      2540

    accuracy                          0.656     15874
   macro avg      0.687     0.687     0.687     15874
weighted avg      0.658     0.656     0.657     15874


=== arousal operating curve, within CREMA-D ===
 threshold  fear_recall  fa_within
    0.0000        0.922      0.420
    0.0001        0.911      0.397
    0.0001        0.899      0.375
    0.0002        0.890      0.353
    0.0004        0.874      0.331
    0.0006        0.859      0.310
    0.0012        0.840      0.289
    0.0021        0.826      0.268
    0.0038        0.807      0.247
    

In [6]:
# ===== CELL 4b — arousal threshold against medical speech =====
rows = []
for t in np.quantile(oof_proba[is_fear], np.linspace(0.05, 0.9, 20)):
    r = {"threshold": round(float(t), 4),
         "fear_recall": round(float((oof_proba[is_fear] >= t).mean()), 3),
         "fa_within_crema": round(float((oof_proba[~is_fear] >= t).mean()), 3)}
    for cond in ["clean", "aug"]:
        s = arousal_final.predict_proba(F[("med", cond)])[:, fear_index_new]
        r[f"fa_medical_{cond}"] = round(float((s >= t).mean()), 4)
    for name in ["esc", "synth"]:
        s = arousal_final.predict_proba(F[(name, "aug")])[:, fear_index_new]
        r[f"fa_{name}"] = round(float((s >= t).mean()), 3)
    rows.append(r)
acurve2 = pd.DataFrame(rows)
print(acurve2.to_string(index=False))

 threshold  fear_recall  fa_within_crema  fa_medical_clean  fa_medical_aug  fa_esc  fa_synth
    0.0000        0.950            0.507            0.2050          0.3162   0.004       0.0
    0.0001        0.905            0.388            0.1588          0.2387   0.000       0.0
    0.0006        0.860            0.312            0.1212          0.2013   0.000       0.0
    0.0030        0.815            0.254            0.0938          0.1800   0.000       0.0
    0.0101        0.771            0.214            0.0862          0.1588   0.000       0.0
    0.0291        0.726            0.177            0.0762          0.1475   0.000       0.0
    0.0668        0.681            0.150            0.0712          0.1375   0.000       0.0
    0.1526        0.637            0.124            0.0612          0.1200   0.000       0.0
    0.2977        0.592            0.103            0.0550          0.1075   0.000       0.0
    0.4987        0.547            0.083            0.0500          0.

In [8]:
# ===== CELL 5 — save =====
import joblib

DISTRESS_HIGH_NEW = 0.4028
NON_SPEECH_HIGH   = 0.50          # from the arousal head's 7th class
non_speech_index  = names.index("non_speech")

joblib.dump({"model": distress_final, "threshold": DISTRESS_HIGH_NEW,
             "extraction": "no_pad_meanpool_rms0.05",
             "negatives": "vivae other + crema neutral + esc50 + silence",
             "heldout_auc": round(float(roc_auc_score(yall, oof2)), 4)},
            "/kaggle/working/distress_gate.joblib")

joblib.dump({"model": arousal_final, "classes": names,
             "fear_index": fear_index_new,
             "non_speech_index": non_speech_index,
             "non_speech_threshold": NON_SPEECH_HIGH,
             "escalates": False,
             "note": "fear probability is displayed, not used for escalation: "
                     "14.5% recall at a 2% medical false-alarm budget. The "
                     "non_speech class (F1 0.988) serves as the speech guard."},
            "/kaggle/working/arousal_head.joblib")

dep.to_csv("/kaggle/working/operating_curve_distress_v2.csv", index=False)
acurve2.to_csv("/kaggle/working/operating_curve_arousal_v2.csv", index=False)

summary = {
    "distress_heldout_auc_all_negatives": round(float(roc_auc_score(yall, oof2)), 4),
    "distress_withincorpus_auc": round(float(auc_in), 4),
    "DISTRESS_HIGH_old": 0.0677, "DISTRESS_HIGH_new": DISTRESS_HIGH_NEW,
    "distress_recall": 0.637, "distress_fa_medical": 0.0075,
    "distress_fa_ward_noise": 0.017, "distress_fa_silence": 0.0,
    "arousal_macro_f1": 0.687, "arousal_fear_f1": 0.552,
    "arousal_non_speech_f1": 0.988,
    "arousal_escalation": "removed",
    "arousal_recall_at_2pct_medical_fa": 0.145,
    "vivae_duplicates_removed": int(n_before - len(vd)),
}
json.dump(summary, open("/kaggle/working/head_refit_v2.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

{
  "distress_heldout_auc_all_negatives": 0.8632,
  "distress_withincorpus_auc": 0.6293,
  "DISTRESS_HIGH_old": 0.0677,
  "DISTRESS_HIGH_new": 0.4028,
  "distress_recall": 0.637,
  "distress_fa_medical": 0.0075,
  "distress_fa_ward_noise": 0.017,
  "distress_fa_silence": 0.0,
  "arousal_macro_f1": 0.687,
  "arousal_fear_f1": 0.552,
  "arousal_non_speech_f1": 0.988,
  "arousal_escalation": "removed",
  "arousal_recall_at_2pct_medical_fa": 0.145,
  "vivae_duplicates_removed": 480
}
